In [0]:
%sql
create schema if not exists gold

In [0]:
%sql
select * from db_weather_streaming.gold.gold_current_weather

event_id,ingestion_time,city,region,country,lat,lon,localtime,temp_c,weather_condition,wind_kph,wind_degree,wind_dir,pressure_mb,humidity,cloud,feelslike_c,uv,pm2_5,pm10,epa_index,rn,air_quality_status
3acf212e-4a5f-42db-b5f8-4f76aed2ec3f,2026-05-09T05:45:50.428283Z,Chennai,Tamil Nadu,India,13.0833,80.2833,2026-05-09T11:13:00Z,30.1,Mist,16.9,74,ENE,1009.0,84,75,34.0,11.7,5.65,6.65,1,1,Low


In [0]:
%sql
CREATE OR REPLACE TABLE db_weather_streaming.gold.gold_current_weather AS
SELECT *,
  CASE 
    WHEN epa_index <= 2 THEN 'Low'
    WHEN epa_index <= 4 THEN 'Moderate'
    ELSE 'High'
  END AS air_quality_status
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY city ORDER BY ingestion_time DESC) AS rn
    FROM db_weather_streaming.silver.silver_weather
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE db_weather_streaming.gold.gold_weather_trends AS

SELECT

    city,

    localtime,

    temp_c,

    feelslike_c,

    humidity,

    pressure_mb,

    wind_kph,

    uv,

    pm2_5,

    pm10

FROM db_weather_streaming.silver.silver_weather;

num_affected_rows,num_inserted_rows


In [0]:
# %sql
# CREATE OR REPLACE TABLE db_weather_streaming.gold.gold_weather_trends AS
# SELECT
#     city,
#     date_trunc('minute', ingestion_time) AS time_window,

#     AVG(temp_c) AS temp_c,
#     AVG(feelslike_c) AS feelslike_c,
#     AVG(pressure_mb) AS pressure_mb,
#     AVG(humidity) AS humidity,
#     AVG(wind_kph) AS wind_kph,
#     AVG(uv) AS uv

# FROM db_weather_streaming.silver.silver_weather
# GROUP BY
#     city,
#     date_trunc('minute', ingestion_time);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from db_weather_streaming.gold.gold_weather_trends

city,localtime,temp_c,feelslike_c,humidity,pressure_mb,wind_kph,uv,pm2_5,pm10
Chennai,2026-05-09T11:13:00Z,30.1,34.0,84,1009.0,16.9,11.7,5.65,6.65
Chennai,2026-05-09T11:13:00Z,30.1,34.0,84,1009.0,16.9,11.7,5.65,6.65
Chennai,2026-05-09T11:13:00Z,30.1,34.0,84,1009.0,16.9,11.7,5.65,6.65
Chennai,2026-05-09T11:13:00Z,30.1,34.0,84,1009.0,16.9,11.7,5.65,6.65
Chennai,2026-05-09T11:13:00Z,30.1,34.0,84,1009.0,16.9,11.7,5.65,6.65


In [0]:
%sql
CREATE OR REPLACE TABLE db_weather_streaming.gold.gold_forecast AS

SELECT *

FROM (

    SELECT
        city,
        forecast_date,
        max_temp_c,
        min_temp_c,
        forecast_condition,
        ingestion_time,

        ROW_NUMBER() OVER (

            PARTITION BY city, forecast_date

            ORDER BY ingestion_time DESC

        ) AS rn

    FROM db_weather_streaming.silver.silver_forecast

)

WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from db_weather_streaming.gold.gold_forecast

city,forecast_date,max_temp_c,min_temp_c,forecast_condition,ingestion_time,rn
Chennai,2026-05-09,31.4,28.5,Patchy rain nearby,2026-05-09T05:45:50.428283Z,1
Chennai,2026-05-10,30.8,28.0,Patchy rain nearby,2026-05-09T05:45:50.428283Z,1
Chennai,2026-05-11,31.1,28.1,Sunny,2026-05-09T05:45:50.428283Z,1
